# OpenAQ data acquisition
MATH70076 Assessment 2 — Question 1

Run the cells in order. Run `pip install -e ".[dev]"` in the project root first.


## 0. Dependencies

Run once. If you already have these, skip it.

In [ ]:
%pip install -e "..[dev]"

## 1. Import the module

The acquisition logic lives in `src/openaq_survey/client.py` rather than in this notebook,
so it can be tested, reused and cited as evidence. This notebook is the
narrative; the module is the tool.

`%autoreload` means edits to the .py take effect without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import pandas as pd

from openaq_survey import client as oaq

DATA = Path("..") / "data"
DATA.mkdir(exist_ok=True)
print("ready")

## 2. API key

Running the next cell pops up a hidden input box. Paste your key and press Enter.

The key is held in memory only — it is **not** written into this notebook file,
so the notebook is safe to commit to a public repo.

In [ ]:
oaq.set_api_key()

## 3. Check the connection

A known station (New Delhi, id 8118) from the OpenAQ quick-start docs.

In [ ]:
probe = oaq.get("/locations/8118")
print(json.dumps(probe["results"][0], indent=2, ensure_ascii=False)[:1200])

## 4. What pollutants exist?

Gives the numeric `parameter_id` values used elsewhere in the API.

In [ ]:
params = oaq.get("/parameters", {"limit": 100})["results"]
pd.DataFrame(params)[["id", "name", "units", "displayName"]].sort_values("id")

## 5. Fetch the station inventory

This sweeps every monitoring station on the platform. **Expect several minutes.**

The result is cached to `data/raw_locations.json`. Re-running this cell after the
first time reads the cache and makes no network calls, which is what lets the
analysis be reproduced offline.

If you see `[429] rate limited` lines — screenshot them. That is direct evidence
for the reflection on rate limits.

In [ ]:
records = oaq.cached_json(
    DATA / "raw_locations.json",
    lambda: list(oaq.paginate("/locations")),
)
print(f"\n{len(records):,} station records")

### Inspect one raw record

Before flattening, look at what the API actually returns. If the flattening in
step 6 produces all-null columns, the field names differ from what was assumed —
this cell is where you find that out.

In [ ]:
print(json.dumps(records[0], indent=2, ensure_ascii=False))

## 6. Flatten to tidy tables

- `stations` — one row per monitoring station
- `sensors` — one row per station × pollutant

In [ ]:
stations = oaq.flatten_stations(records)
sensors  = oaq.flatten_sensors(records)

stations.to_csv(DATA / "stations.csv", index=False)
sensors.to_csv(DATA / "sensors.csv", index=False)

print(f"stations: {stations.shape}   sensors: {sensors.shape}")
stations.head()

### Did anything fail to parse?

Any column at 1.00 means the field name guess was wrong — check the raw record above.

In [ ]:
stations.isna().mean().sort_values(ascending=False).to_frame("null_fraction")

## 7. Summary

These are the numbers that determine what argument the data can support.

In [ ]:
print(f"stations        : {len(stations):,}")
print(f"sensors         : {len(sensors):,}")
print(f"countries       : {stations['country_code'].nunique()}")
print(f"earliest record : {stations['datetime_first'].min()}")
print(f"latest record   : {stations['datetime_last'].max()}")

print("\nreference monitors vs other:")
print(stations["is_monitor"].value_counts(dropna=False))

print("\ntop 15 countries by station count:")
print(stations["country_code"].value_counts().head(15))

print("\nfewest stations:")
print(stations["country_code"].value_counts().tail(15))

print("\npollutant coverage:")
print(sensors["parameter"].value_counts().head(15))

print("\nlicences:")
print(stations["licence"].value_counts(dropna=False).head(10))

print("\ndata providers:")
print(stations["provider"].value_counts().head(10))

## 8. Per-country table

The unit of analysis for the argument about monitoring coverage.

In [ ]:
by_country = (
    stations
    .groupby(["country_code", "country_name"], dropna=False)
    .agg(
        n_stations=("location_id", "count"),
        n_reference=("is_monitor", "sum"),
        n_sensors=("n_sensors", "sum"),
        earliest=("datetime_first", "min"),
    )
    .reset_index()
    .sort_values("n_stations", ascending=False)
)
by_country.to_csv(DATA / "by_country.csv", index=False)
print(f"{len(by_country)} countries")
by_country.head(30)

## 9. Before committing

Notebook outputs are saved inside the `.ipynb` file. Clear them before pushing,
or install `nbstripout` to do it automatically on every commit:

```
pip install nbstripout
nbstripout --install
```

Keep a note as you go of anything that broke, surprised you, or made you change
approach — that is the raw material for the reflective writing, and it will not
be recoverable from memory in three days.
